In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from types import SimpleNamespace
from tqdm import tqdm
import itertools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class Instance:
    def __init__(self, text, label):
        self.text = text.split(' ')
        self.label = label

In [ ]:
class Vocab:
    def __init__(self, frequency, max_size, min_freq):
      self.stoi = {}
      self.itos = {}

      frequency = sorted(frequency.items(), key=lambda x: x[1], reverse=True)
      for i, (w, f) in enumerate(frequency):
        if f >= min_freq and (len(self.stoi) < max_size or max_size == -1):
          self.stoi[w] = i
          self.itos[i] = w
        else:
          break

    def __len__(self):
      return len(self.stoi)

    def encode(self, tokens):
      if isinstance(tokens, str):
        return torch.tensor(self.stoi.get(tokens, self.stoi.get('<UNK>', 0)))
      return torch.tensor([self.stoi.get(token, self.stoi.get('<UNK>', 0)) for token in tokens])

In [ ]:
def calc_frequencies(data):
  word_counter = {}
  label_counter = {}

  for d in data:
    for word in d.text:
      word_counter[word] = word_counter.get(word, 0) + 1
    label_counter[d.label] = label_counter.get(d.label, 0) + 1
  return word_counter, label_counter

In [ ]:
class NPLDataset(torch.utils.data.Dataset):
    def __init__(self, path, split="train", max_size=-1, min_freq=0):
        self.data = []
        csv = pd.read_csv(path, sep=', ', header=None)

        for _, row in csv.iterrows():
            self.data.append(Instance(row[0], row[1]))

        if split == "train":
          self.frequency = calc_frequencies(self.data)
          self.frequency[0]['<PAD>'] = float('inf')
          self.frequency[0]['<UNK>'] = float('inf')
          self.text_vocab = Vocab(self.frequency[0], max_size, min_freq)
          self.label_vocab = Vocab(self.frequency[1], -1, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.text_vocab.encode(self.data[index].text), self.label_vocab.encode(self.data[index].label)

In [ ]:
d = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
d.data[3].text

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


['yet', 'the', 'act', 'is', 'still', 'charming', 'here']

In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab, path, embedding_dim=300):
    super(Embedding, self).__init__()
    self.embedding = nn.Embedding(len(vocab), embedding_dim, padding_idx=vocab.stoi['<PAD>'])

    embed = {}
    if path is not None:
      with open(path) as f:
        for i in f:
          word, vector = i.split(' ', 1)
          embed[word] = torch.tensor([float(x) for x in vector.split()])

    for i, word in vocab.itos.items():
      if word == '<PAD>':
        self.embedding.weight.data[i] = torch.zeros(embedding_dim)
      elif word in embed:
        self.embedding.weight.data[i] = embed[word]

    self.embedding.freeze = path is not None

  def forward(self, x):
    return self.embedding(x)

In [ ]:
def pad_collate_fn(batch, pad_index=0):
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(text) for text in texts])

    max_length = lengths.max()
    padded = torch.stack([F.pad(text, (0, max_length - len(text)), value=pad_index) for text in texts])
    return padded, torch.tensor(labels), lengths

In [ ]:
train_dataldr = DataLoader(dataset=d, batch_size=2, shuffle=False, collate_fn=pad_collate_fn)
texts, labels, lengths = next(iter(train_dataldr))
print(f"Texts: {texts}")
print(f"Labels: {labels}")
print(f"Lengths: {lengths}")

Texts: tensor([[   2,  554,    7, 2872,    6,   22,    2, 2873, 1236,    8,   96, 4800,
            4,   10,   72,    8,  242,    6,   75,    3, 3576,   56, 3577,   34,
         2022, 2874, 7123, 3578, 7124,   42,  779, 7125,    0,    0],
        [   2, 2875, 2023, 4801,    5,    2, 3579,    5,    2, 2876, 4802,    7,
           40,  829,   10,    3, 4803,    5,  627,   62,   27, 2877, 2024, 4804,
          962,  715,    8, 7126,  555,    5, 7127, 4805,    8, 7128]])
Labels: tensor([0, 0])
Lengths: tensor([32, 34])


In [ ]:
class Model(nn.Module):
  def __init__(self, embedding, embedding_dim=300):
    super().__init__()
    self.embedding = embedding
    self.fc1 = nn.Linear(embedding_dim, 150)
    self.fc2 = nn.Linear(150, 150)
    self.fc3 = nn.Linear(150, 1)

  def forward(self, x):
    emb = self.embedding(x)
    pool = F.avg_pool1d(emb.permute(0, 2, 1), emb.shape[1]).squeeze(2)
    x = F.relu(self.fc1(pool))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    x = x.squeeze(1)
    return x

In [ ]:
def train(model, data, optimizer, criterion, args):
  total_loss = 0
  model.train()

  with tqdm(total=len(data)) as pbar:
    pbar.set_description("Training")
    for batch_num, batch in enumerate(data):
      x, y, _ = batch
      x, y = x.to(args.device), y.to(args.device).float()

      optimizer.zero_grad()
      logits = model(x)
      loss = criterion(logits, y.float())
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
      optimizer.step()
      total_loss = total_loss + loss.item()
      pbar.set_postfix(loss=loss.item())
      pbar.update(1)
  return total_loss / len(data)


def evaluate(model, data, criterion, args):
  total_loss = 0
  all_labels = []
  all_preds = []

  model.eval()

  with torch.no_grad():
    for batch_num, batch in enumerate(data):
      x, y, _ = batch
      x, y = x.to(args.device), y.to(args.device).float()
      logits = model(x)
      loss = criterion(logits, y.float())
      total_loss = total_loss + loss.item()

      predictions = torch.sigmoid(logits) >= 0.5
      all_labels.extend(predictions.cpu().numpy())
      all_preds.extend(y.cpu().numpy())

  avg_loss = total_loss / len(data)
  accuracy = accuracy_score(all_labels, all_preds)
  f1 = f1_score(all_labels, all_preds)
  cm = confusion_matrix(all_labels, all_preds)

  print(f"Loss: {avg_loss}")
  print(f"Accuracy: {accuracy}")
  print(f"F1: {f1}")
  print(f"Confusion Matrix: \n{cm}")

  return avg_loss, accuracy, f1, cm

In [ ]:
def main(args):
  seed = args.seed

  train_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
  valid_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_valid_raw.csv')
  test_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_test_raw.csv')
  vocab = train_dataset.text_vocab
  valid_dataset.text_vocab = vocab
  test_dataset.text_vocab = vocab
  valid_dataset.label_vocab = train_dataset.label_vocab
  test_dataset.label_vocab = train_dataset.label_vocab

  train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=pad_collate_fn)
  valid_dataloader = DataLoader(dataset=valid_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
  test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)

  embedding = Embedding(vocab, '/content/drive/MyDrive/Colab Notebooks/sst_glove_6b_300d.txt', args.embedding_dim)
  np.random.seed(seed)
  torch.manual_seed(seed)
  model = Model(embedding, embedding_dim=args.embedding_dim)
  model.to(args.device)

  criterion = nn.BCEWithLogitsLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

  for epoch in range(args.epochs):
    print(f"Epoch: {epoch+1}")
    train(model, train_dataloader, optimizer, criterion, args)
    evaluate(model, valid_dataloader, criterion, args)

  print("Testing model")
  evaluate(model, test_dataloader, criterion, args)

In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "embedding_dim": 300,
    "epochs": 5,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 1.0
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Epoch: 1


Training: 100%|██████████| 692/692 [00:08<00:00, 81.05it/s, loss=0.709] 


Loss: 0.5737520377886923
Accuracy: 0.7534321801208127
F1: 0.7369654364381957
Confusion Matrix: 
[[743 283]
 [166 629]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:06<00:00, 103.97it/s, loss=0.495]


...

Loss: 0.4736872323249516
Accuracy: 0.7830862163646348
F1: 0.7472808701215611
Confusion Matrix: 
[[842 328]
 [ 67 584]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:04<00:00, 170.28it/s, loss=0.48]


Loss: 0.417522920589698
Accuracy: 0.814936847885777
F1: 0.8137092316196793
Confusion Matrix: 
[[748 176]
 [161 736]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 163.08it/s, loss=0.332]


Loss: 0.42897985222046836
Accuracy: 0.8094453596924767
F1: 0.7993059572006941
Confusion Matrix: 
[[783 221]
 [126 691]]
Testing model
Loss: 0.4651187466723578
Accuracy: 0.7970183486238532
F1: 0.7849331713244229
Confusion Matrix: 
[[372 105]
 [ 72 323]]


In [ ]:
class RNN(nn.Module):
  def __init__(self, type_, embedding, input_size, hidden_size, num_layers, bidirectional, dropout, batch_first=True):
    super(RNN, self).__init__()
    self.embedding = embedding
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.bidirectiona = bidirectional

    if type_ == 'LSTM':
      self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=batch_first, bidirectional=bidirectional, dropout=dropout)
    elif type_ == 'GRU':
      self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=batch_first, bidirectional=bidirectional, dropout=dropout)
    elif type_ == 'RNN':
      self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=batch_first, bidirectional=bidirectional, dropout=dropout)

    fc_input = hidden_size * 2 if bidirectional else hidden_size

    self.fc1 = nn.Linear(fc_input, hidden_size)
    self.fc2 = nn.Linear(hidden_size, 1)

  def forward(self, x):
    emb = self.embedding(x)
    hidden, _ = self.rnn(emb)
    hidden = hidden[:, -1, :]

    x = self.fc1(hidden)
    x = F.relu(x)
    x = self.fc2(x).squeeze(1)
    return x

In [ ]:
def main(args):
  seed = args.seed

  model_type = ['LSTM', 'GRU', 'RNN']

  train_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
  valid_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_valid_raw.csv')
  test_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_test_raw.csv')
  vocab = train_dataset.text_vocab
  valid_dataset.text_vocab = vocab
  test_dataset.text_vocab = vocab
  valid_dataset.label_vocab = train_dataset.label_vocab
  test_dataset.label_vocab = train_dataset.label_vocab

  train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=pad_collate_fn)
  valid_dataloader = DataLoader(dataset=valid_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
  test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)

  embedding = Embedding(vocab, '/content/drive/MyDrive/Colab Notebooks/sst_glove_6b_300d.txt', args.embedding_dim)
  np.random.seed(seed)
  torch.manual_seed(seed)

  for type_ in model_type:
    print(f"Model type: {type_}")
    model = RNN(type_, embedding=embedding, input_size=args.embedding_dim, hidden_size=args.hidden_size, num_layers=args.num_layers, bidirectional=args.bidirectional, dropout=args.dropout)
    model.to(args.device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(args.epochs):
      print(f"Epoch: {epoch+1}")
      train(model, train_dataloader, optimizer, criterion, args)
      evaluate(model, valid_dataloader, criterion, args)

    print("Testing model")
    evaluate(model, test_dataloader, criterion, args)

In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:06<00:00, 99.36it/s, loss=0.316]


Loss: 0.5935396523329249
Accuracy: 0.7457440966501923
F1: 0.7026332691072575
Confusion Matrix: 
[[811 365]
 [ 98 547]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:06<00:00, 99.36it/s, loss=0.348] 


...

Loss: 0.8877397707679815
Accuracy: 0.7957166392092258
F1: 0.7995689655172413
Confusion Matrix: 
[[707 170]
 [202 742]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:05<00:00, 131.70it/s, loss=0.0221]


Loss: 1.0462715861044432
Accuracy: 0.7830862163646348
F1: 0.7587049480757483
Confusion Matrix: 
[[805 291]
 [104 621]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 146.24it/s, loss=1.11]


Loss: 1.0092716674532807
Accuracy: 0.8017572762218561
F1: 0.7904817179338364
Confusion Matrix: 
[[779 231]
 [130 681]]
Testing model
Loss: 1.0529521267328943
Accuracy: 0.7935779816513762
F1: 0.7777777777777778
Confusion Matrix: 
[[377 113]
 [ 67 315]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 50,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:05<00:00, 123.43it/s, loss=0.685]


Loss: 0.6939139292951215
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:05<00:00, 125.62it/s, loss=0.646]


...

Loss: 0.5727606698086387
Accuracy: 0.7682591982427238
F1: 0.7877263581488934
Confusion Matrix: 
[[616 129]
 [293 783]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:05<00:00, 129.46it/s, loss=1.29]


Loss: 0.6087913426913714
Accuracy: 0.7808896210873146
F1: 0.7874267448055408
Confusion Matrix: 
[[683 173]
 [226 739]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 147.84it/s, loss=0.403]


Loss: 0.7320123207673692
Accuracy: 0.7781438769906645
F1: 0.798
Confusion Matrix: 
[[619 114]
 [290 798]]
Testing model
Loss: 0.7394102332847459
Accuracy: 0.7786697247706422
F1: 0.7948990435706695
Confusion Matrix: 
[[305  54]
 [139 374]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 450,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:09<00:00, 74.37it/s, loss=0.618]


Loss: 0.5058038663445857
Accuracy: 0.7457440966501923
F1: 0.7851508120649652
Confusion Matrix: 
[[512  66]
 [397 846]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:09<00:00, 72.06it/s, loss=0.334]


...

Loss: 1.7497903098139846
Accuracy: 0.700164744645799
F1: 0.6002928257686676
Confusion Matrix: 
[[865 502]
 [ 44 410]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:07<00:00, 90.85it/s, loss=0.508] 


Loss: 1.7617985993100886
Accuracy: 0.7259747391543108
F1: 0.6815571155073389
Confusion Matrix: 
[[788 378]
 [121 534]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:06<00:00, 100.99it/s, loss=0.00924]


Loss: 1.26772285082884
Accuracy: 0.6968698517298187
F1: 0.6051502145922747
Confusion Matrix: 
[[846 489]
 [ 63 423]]
Testing model
Loss: 1.2821840510836668
Accuracy: 0.6972477064220184
F1: 0.6094674556213018
Confusion Matrix: 
[[402 222]
 [ 42 206]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 2,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:06<00:00, 109.93it/s, loss=0.897]


Loss: 0.5932461405009554
Accuracy: 0.71444261394838
F1: 0.7031963470319634
Confusion Matrix: 
[[685 296]
 [224 616]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:05<00:00, 120.99it/s, loss=0.299]


...

Loss: 0.9667350030259082
Accuracy: 0.7918725974739155
F1: 0.7807981492192019
Confusion Matrix: 
[[767 237]
 [142 675]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:05<00:00, 135.76it/s, loss=0.465]


Loss: 0.9799162538950903
Accuracy: 0.7902251510159253
F1: 0.7771295215869312
Confusion Matrix: 
[[773 246]
 [136 666]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 145.45it/s, loss=0.0045]


Loss: 1.100050958886481
Accuracy: 0.7913234486545854
F1: 0.7775175644028103
Confusion Matrix: 
[[777 248]
 [132 664]]
Testing model
Loss: 1.1836470542475581
Accuracy: 0.7763761467889908
F1: 0.7619047619047619
Confusion Matrix: 
[[365 116]
 [ 79 312]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 15,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:15<00:00, 43.31it/s, loss=0.701]


Loss: 0.6939392037558973
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:16<00:00, 42.78it/s, loss=0.687]


...

Loss: 0.6941579015631425
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:07<00:00, 93.48it/s, loss=0.734]


Loss: 0.6959369956401357
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:07<00:00, 96.09it/s, loss=0.689] 


Loss: 0.694385760708859
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Testing model
Loss: 0.6946718373468944
Accuracy: 0.5091743119266054
F1: 0.0
Confusion Matrix: 
[[444 428]
 [  0   0]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": True,
    "dropout": 0.25,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:09<00:00, 73.35it/s, loss=0.358]


Loss: 0.5048997057111639
Accuracy: 0.7616694124107634
F1: 0.7887049659201558
Confusion Matrix: 
[[577 102]
 [332 810]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:09<00:00, 72.48it/s, loss=0.387]


...

Loss: 0.8449916983382744
Accuracy: 0.8116419549697969
F1: 0.8210745957224831
Confusion Matrix: 
[[691 125]
 [218 787]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:05<00:00, 128.19it/s, loss=0.00268]


Loss: 1.0964904417071426
Accuracy: 0.7984623833058759
F1: 0.7943977591036414
Confusion Matrix: 
[[745 203]
 [164 709]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:05<00:00, 115.54it/s, loss=0.00481]


Loss: 0.9992258187971617
Accuracy: 0.8012081274025261
F1: 0.8139773895169579
Confusion Matrix: 
[[667 120]
 [242 792]]
Testing model
Loss: 1.0685373087014471
Accuracy: 0.7889908256880734
F1: 0.7973568281938326
Confusion Matrix: 
[[326  66]
 [118 362]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 0,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:07<00:00, 94.97it/s, loss=0.5] 


Loss: 0.6493921005412152
Accuracy: 0.7473915431081823
F1: 0.7181372549019608
Confusion Matrix: 
[[775 326]
 [134 586]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:06<00:00, 106.83it/s, loss=0.406]


...

Loss: 1.0563116078836876
Accuracy: 0.7737506864360242
F1: 0.7541766109785203
Confusion Matrix: 
[[777 280]
 [132 632]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:04<00:00, 141.56it/s, loss=0.00893]


Loss: 1.2407987297496252
Accuracy: 0.7391543108182317
F1: 0.6744345442083619
Confusion Matrix: 
[[854 420]
 [ 55 492]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 144.78it/s, loss=0.533]


Loss: 1.0479688171231956
Accuracy: 0.7979132344865458
F1: 0.805702217529039
Confusion Matrix: 
[[690 149]
 [219 763]]
Testing model
Loss: 1.127778603479133
Accuracy: 0.7775229357798165
F1: 0.780045351473923
Confusion Matrix: 
[[334  84]
 [110 344]]


In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "hidden_size": 150,
    "num_layers": 3,
    "embedding_dim": 300,
    "epochs": 5,
    "bidirectional": False,
    "dropout": 1,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

main(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:07<00:00, 93.91it/s, loss=0.685]


Loss: 0.6943989301982679
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:06<00:00, 106.86it/s, loss=0.68]


...

Loss: 0.6951330513284918
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 4


Training: 100%|██████████| 692/692 [00:05<00:00, 130.87it/s, loss=0.685]


Loss: 0.6945508519808451
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Epoch: 5


Training: 100%|██████████| 692/692 [00:04<00:00, 144.12it/s, loss=0.704]


Loss: 0.6946886370056554
Accuracy: 0.49917627677100496
F1: 0.0
Confusion Matrix: 
[[909 912]
 [  0   0]]
Testing model
Loss: 0.6948226903166089
Accuracy: 0.5091743119266054
F1: 0.0
Confusion Matrix: 
[[444 428]
 [  0   0]]


In [ ]:
def grid_search(args):
  seed = args.seed

  model_type = ['LSTM']
  hidden_size = [150, 300]
  num_layers = [3, 10]
  bidirectional = [True, False]
  dropout = [0.05, 0.25]
  best_f1 = 0
  best_model_state = None
  best_params = None

  train_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
  valid_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_valid_raw.csv')
  test_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_test_raw.csv')
  vocab = train_dataset.text_vocab
  valid_dataset.text_vocab = vocab
  test_dataset.text_vocab = vocab
  valid_dataset.label_vocab = train_dataset.label_vocab
  test_dataset.label_vocab = train_dataset.label_vocab

  train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=pad_collate_fn)
  valid_dataloader = DataLoader(dataset=valid_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
  test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)

  embedding = Embedding(vocab, '/content/drive/MyDrive/Colab Notebooks/sst_glove_6b_300d.txt', args.embedding_dim)
  np.random.seed(seed)
  torch.manual_seed(seed)

  for type_ in model_type:
    print(f"Model type: {type_}")
    for hs, nl, bi, dp in itertools.product(hidden_size, num_layers, bidirectional, dropout):
      model = RNN(type_, embedding=embedding, input_size=args.embedding_dim, hidden_size=hs, num_layers=nl, bidirectional=bi, dropout=dp)
      model.to(args.device)

      criterion = nn.BCEWithLogitsLoss()
      optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

      for epoch in range(args.epochs):
        print(f"Epoch: {epoch+1}")
        train(model, train_dataloader, optimizer, criterion, args)
        evaluate(model, valid_dataloader, criterion, args)

      val_loss, val_acc, val_f1, val_cm = evaluate(model, valid_dataloader, criterion, args)

      if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict()
            best_params = {
                'model_type': type_,
                'hidden_size': hs,
                'num_layers': nl,
                'dropout': dp,
                'bidirectional': bi
            }
            torch.save({'model_state_dict': best_model_state, 'params': best_params}, 'best_params_LSTM.pt')

    print("Testing model")
    evaluate(model, test_dataloader, criterion, args)

  print("\n Grid search finished.")
  print(f"Best F1-score: {best_f1:.4f}")
  print(f"Best hyperparameters: {best_params}")

In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "embedding_dim": 300,
    "epochs": 3,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

grid_search(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: LSTM
Epoch: 1


Training: 100%|██████████| 692/692 [00:09<00:00, 71.93it/s, loss=0.391]


Loss: 0.5515666734753993
Accuracy: 0.7462932454695222
F1: 0.7820754716981132
Confusion Matrix: 
[[530  83]
 [379 829]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:09<00:00, 73.15it/s, loss=0.318]


...

Loss: 1.5375832290503018
Accuracy: 0.7726523887973641
F1: 0.7960591133004926
Confusion Matrix: 
[[599 104]
 [310 808]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:16<00:00, 42.13it/s, loss=0.00125]


Loss: 1.7286892402590366
Accuracy: 0.7616694124107634
F1: 0.7872549019607843
Confusion Matrix: 
[[584 109]
 [325 803]]
Epoch: 3


Training: 100%|██████████| 692/692 [00:16<00:00, 42.34it/s, loss=0.000646]


Loss: 1.8895217807669389
Accuracy: 0.7545304777594728
F1: 0.7874465049928673
Confusion Matrix: 
[[546  84]
 [363 828]]
Loss: 1.8895217807669389
Accuracy: 0.7545304777594728
F1: 0.7874465049928673
Confusion Matrix: 
[[546  84]
 [363 828]]
Testing model
Loss: 1.9319967227477588
Accuracy: 0.7477064220183486
F1: 0.779559118236473
Confusion Matrix: 
[[263  39]
 [181 389]]

 Grid search finished.
Best F1-score: 0.8203
Best hyperparameters: {'model_type': 'LSTM', 'hidden_size': 150, 'num_layers': 10, 'dropout': 0.25, 'bidirectional': True}


In [ ]:
def grid_search(args):
  seed = args.seed

  model_type = ['GRU']
  hidden_size = [150, 300]
  num_layers = [3, 10]
  bidirectional = [True, False]
  dropout = [0.05, 0.25]
  best_f1 = 0
  best_model_state = None
  best_params = None

  train_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
  valid_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_valid_raw.csv')
  test_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_test_raw.csv')
  vocab = train_dataset.text_vocab
  valid_dataset.text_vocab = vocab
  test_dataset.text_vocab = vocab
  valid_dataset.label_vocab = train_dataset.label_vocab
  test_dataset.label_vocab = train_dataset.label_vocab

  train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=pad_collate_fn)
  valid_dataloader = DataLoader(dataset=valid_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
  test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)

  embedding = Embedding(vocab, '/content/drive/MyDrive/Colab Notebooks/sst_glove_6b_300d.txt', args.embedding_dim)
  np.random.seed(seed)
  torch.manual_seed(seed)

  for type_ in model_type:
    print(f"Model type: {type_}")
    for hs, nl, bi, dp in itertools.product(hidden_size, num_layers, bidirectional, dropout):
      model = RNN(type_, embedding=embedding, input_size=args.embedding_dim, hidden_size=hs, num_layers=nl, bidirectional=bi, dropout=dp)
      model.to(args.device)

      criterion = nn.BCEWithLogitsLoss()
      optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

      for epoch in range(args.epochs):
        print(f"Epoch: {epoch+1}")
        train(model, train_dataloader, optimizer, criterion, args)
        evaluate(model, valid_dataloader, criterion, args)

      val_loss, val_acc, val_f1, val_cm = evaluate(model, valid_dataloader, criterion, args)

      if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict()
            best_params = {
                'model_type': type_,
                'hidden_size': hs,
                'num_layers': nl,
                'dropout': dp,
                'bidirectional': bi
            }
            torch.save({'model_state_dict': best_model_state, 'params': best_params}, 'best_params_GRU.pt')

    print("Testing model")
    evaluate(model, test_dataloader, criterion, args)

  print("\n Grid search finished.")
  print(f"Best F1-score: {best_f1:.4f}")
  print(f"Best hyperparameters: {best_params}")

In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "embedding_dim": 300,
    "epochs": 3,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

grid_search(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: GRU
Epoch: 1


Training: 100%|██████████| 692/692 [00:08<00:00, 78.20it/s, loss=0.285]


Loss: 0.49866078401866715
Accuracy: 0.7836353651839648
F1: 0.7888531618435155
Confusion Matrix: 
[[691 176]
 [218 736]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:09<00:00, 75.73it/s, loss=0.437]


...

Loss: 1.3635522754568803
Accuracy: 0.8017572762218561
F1: 0.8056004308023694
Confusion Matrix: 
[[712 164]
 [197 748]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:15<00:00, 44.81it/s, loss=0.000303]


Loss: 1.6660230902203343
Accuracy: 0.7803404722679846
F1: 0.7536945812807881
Confusion Matrix: 
[[809 300]
 [100 612]]
Epoch: 3


Training: 100%|██████████| 692/692 [00:15<00:00, 44.68it/s, loss=0.000304]


Loss: 1.5973374623511183
Accuracy: 0.7946183415705657
F1: 0.7812865497076024
Confusion Matrix: 
[[779 244]
 [130 668]]
Loss: 1.5973374623511183
Accuracy: 0.7946183415705657
F1: 0.7812865497076024
Confusion Matrix: 
[[779 244]
 [130 668]]
Testing model
Loss: 1.750489882592644
Accuracy: 0.7752293577981652
F1: 0.755
Confusion Matrix: 
[[374 126]
 [ 70 302]]

 Grid search finished.
Best F1-score: 0.8249
Best hyperparameters: {'model_type': 'GRU', 'hidden_size': 150, 'num_layers': 10, 'dropout': 0.25, 'bidirectional': False}


In [ ]:
def grid_search(args):
  seed = args.seed

  model_type = ['RNN']
  hidden_size = [150, 300]
  num_layers = [3, 10]
  bidirectional = [True, False]
  dropout = [0.05, 0.25]
  best_f1 = 0
  best_model_state = None
  best_params = None

  train_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_train_raw.csv')
  valid_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_valid_raw.csv')
  test_dataset = NPLDataset('/content/drive/MyDrive/Colab Notebooks/sst_test_raw.csv')
  vocab = train_dataset.text_vocab
  valid_dataset.text_vocab = vocab
  test_dataset.text_vocab = vocab
  valid_dataset.label_vocab = train_dataset.label_vocab
  test_dataset.label_vocab = train_dataset.label_vocab

  train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=pad_collate_fn)
  valid_dataloader = DataLoader(dataset=valid_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)
  test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_fn)

  embedding = Embedding(vocab, '/content/drive/MyDrive/Colab Notebooks/sst_glove_6b_300d.txt', args.embedding_dim)
  np.random.seed(seed)
  torch.manual_seed(seed)

  for type_ in model_type:
    print(f"Model type: {type_}")
    for hs, nl, bi, dp in itertools.product(hidden_size, num_layers, bidirectional, dropout):
      model = RNN(type_, embedding=embedding, input_size=args.embedding_dim, hidden_size=hs, num_layers=nl, bidirectional=bi, dropout=dp)
      model.to(args.device)

      criterion = nn.BCEWithLogitsLoss()
      optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

      for epoch in range(args.epochs):
        print(f"Epoch: {epoch+1}")
        train(model, train_dataloader, optimizer, criterion, args)
        evaluate(model, valid_dataloader, criterion, args)

      val_loss, val_acc, val_f1, val_cm = evaluate(model, valid_dataloader, criterion, args)

      if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict()
            best_params = {
                'model_type': type_,
                'hidden_size': hs,
                'num_layers': nl,
                'dropout': dp,
                'bidirectional': bi
            }
            torch.save({'model_state_dict': best_model_state, 'params': best_params}, 'best_params_RNN.pt')

    print("Testing model")
    evaluate(model, test_dataloader, criterion, args)

  print("\n Grid search finished.")
  print(f"Best F1-score: {best_f1:.4f}")
  print(f"Best hyperparameters: {best_params}")

In [ ]:
args = {
    "seed": 7052020,
    "batch_size": 10,
    "embedding_dim": 300,
    "epochs": 3,
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    "clip": 0.25
}

grid_search(SimpleNamespace(**args))

<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)
<ipython-input-6-c9cb7466cede>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  csv = pd.read_csv(path, sep=', ', header=None)


Model type: RNN
Epoch: 1


Training: 100%|██████████| 692/692 [00:06<00:00, 111.74it/s, loss=0.471]


Loss: 0.6955765459621162
Accuracy: 0.6326194398682042
F1: 0.49356548069644207
Confusion Matrix: 
[[826 586]
 [ 83 326]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:05<00:00, 131.45it/s, loss=0.366]


...

Loss: 1.0163533337283552
Accuracy: 0.7973640856672158
F1: 0.7955678670360111
Confusion Matrix: 
[[734 194]
 [175 718]]
Epoch: 2


Training: 100%|██████████| 692/692 [00:11<00:00, 62.69it/s, loss=0.00258]


Loss: 1.1209812446644432
Accuracy: 0.800109829763866
F1: 0.8010928961748633
Confusion Matrix: 
[[724 179]
 [185 733]]
Epoch: 3


Training: 100%|██████████| 692/692 [00:11<00:00, 62.62it/s, loss=0.725]


Loss: 1.9332673889503145
Accuracy: 0.728171334431631
F1: 0.6555323590814196
Confusion Matrix: 
[[855 441]
 [ 54 471]]
Loss: 1.9332673889503145
Accuracy: 0.728171334431631
F1: 0.6555323590814196
Confusion Matrix: 
[[855 441]
 [ 54 471]]
Testing model
Loss: 1.8775914467197643
Accuracy: 0.7362385321100917
F1: 0.6685878962536023
Confusion Matrix: 
[[410 196]
 [ 34 232]]

 Grid search finished.
Best F1-score: 0.7552
Best hyperparameters: {'model_type': 'RNN', 'hidden_size': 150, 'num_layers': 10, 'dropout': 0.05, 'bidirectional': False}
